# Chapter 13

Add your content here.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

# 1. Configuration
# We use d=2 so we can easily plot the geometry 
# We use K=3 classes. A Simplex ETF in 2D for 3 classes is a Mercedes-Benz logo shape (120 degrees apart).
FEAT_DIM = 2 
NUM_CLASSES = 3
SAMPLES_PER_CLASS = 100
EPOCHS = 2000 # Train for a long time to reach "Terminal Phase"

# 2. Create Toy Dataset (Blobs)
# Initial data is just random Gaussian noise centered at different locations
X = []
y = []
for k in range(NUM_CLASSES):
    # Create a blob for class k
    blob = torch.randn(SAMPLES_PER_CLASS, FEAT_DIM) + (torch.randn(1, FEAT_DIM) * 5) 
    X.append(blob)
    y.append(torch.full((SAMPLES_PER_CLASS,), k, dtype=torch.long))

X = torch.cat(X)
y = torch.cat(y)

# 3. Define the Network
# A simple MLP: Input -> Linear -> ReLU -> Linear (Features) -> Linear (Classifier)
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(FEAT_DIM, 50)
        self.relu = nn.ReLU()
        # This is the feature extractor layer (h)
        self.feature_layer = nn.Linear(50, FEAT_DIM, bias=False) 
        # This is the classifier layer (W)
        self.classifier = nn.Linear(FEAT_DIM, NUM_CLASSES, bias=False)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        h = self.feature_layer(x) # These are the features we analyze
        logits = self.classifier(h)
        return h, logits

model = SimpleNet()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

# 4. Training Loop
loss_history = []

print("Training started... watching for Collapse.")
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    features, logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch}: Loss {loss.item():.4f}")

# 5. Analysis: Check for Simplex ETF
# Get the class means of the learned features
with torch.no_grad():
    features, _ = model(X)
    class_means = []
    for k in range(NUM_CLASSES):
        # Extract features belonging to class k
        k_feats = features[y == k]
        mean_k = k_feats.mean(dim=0)
        class_means.append(mean_k)
    
    class_means = torch.stack(class_means)
    
    # Normalize means to check angles
    class_means_norm = class_means / class_means.norm(dim=1, keepdim=True)
    
    # Compute Cosine Similarity Matrix
    gram_matrix = torch.mm(class_means_norm, class_means_norm.t())
    
    print("\n--- Geometry Check ---")
    print(f"Theoretical ETF Cosine for K=3: -1/(3-1) = -0.5")
    print("Actual Cosine Matrix (off-diagonals should be approx -0.5):")
    print(gram_matrix.numpy().round(2))

# Note for Students:
# If successful, the off-diagonal elements (relationship between Class 0 and Class 1, etc.)
# should be very close to -0.5. This proves the network learned a Simplex ETF structure!

Training started... watching for Collapse.
Epoch 0: Loss 0.8971
Epoch 500: Loss 0.0345
Epoch 1000: Loss 0.0243
Epoch 1500: Loss 0.0221

--- Geometry Check ---
Theoretical ETF Cosine for K=3: -1/(3-1) = -0.5
Actual Cosine Matrix (off-diagonals should be approx -0.5):
[[ 1.    0.54 -0.38]
 [ 0.54  1.   -0.98]
 [-0.38 -0.98  1.  ]]
